In [1]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, Flatten, Dense, Dropout, Add, Multiply, GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

2026-07-13 11:09:16.944651: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-13 11:09:16.963741: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-13 11:09:16.963757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-13 11:09:16.964310: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-13 11:09:16.967539: I tensorflow/core/platform/cpu_feature_guar

In [2]:
with h5py.File('processed_physics_data.h5', 'r') as f:
    X_train = f['X_train'][:]
    y_train = f['y_train'][:]
    X_val = f['X_val'][:]
    y_val = f['y_val'][:]
    X_test = f['X_test'][:]
    y_test = f['y_test'][:]

In [3]:
X_train_cnn = np.expand_dims(X_train, -1)
X_val_cnn = np.expand_dims(X_val, -1)
X_test_cnn = np.expand_dims(X_test, -1)

print("--- Notebook B: Deep Learning Environment Initialized ---")
print(f"Train Shape: {X_train_cnn.shape} | Signal: {int(np.sum(y_train))}")
print(f"Val Shape:   {X_val_cnn.shape} | Signal: {int(np.sum(y_val))}")
print(f"Test Shape:  {X_test_cnn.shape} | Signal: {int(np.sum(y_test))}")


--- Notebook B: Deep Learning Environment Initialized ---
Train Shape: (87778, 24, 36, 1) | Signal: 8229
Val Shape:   (21945, 24, 36, 1) | Signal: 2057
Test Shape:  (27431, 24, 36, 1) | Signal: 2572


In [4]:
neg_count = len(y_train) - np.sum(y_train)
pos_count = np.sum(y_train)
total = len(y_train)

weight_for_0 = (1 / neg_count) * (total / 2.0)
weight_for_1 = (1 / pos_count) * (total / 2.0)

class_weights = {0: weight_for_0, 1: weight_for_1}
print(f"\nComputed Class Weights: {class_weights}")


Computed Class Weights: {0: 0.5517228374963858, 1: 5.333454854781869}


In [5]:
#TRAIL 1

In [6]:
inputs = Input(shape=(24, 36, 1))
x = Conv2D(32, (3, 3), padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)

2026-07-13 11:11:47.204405: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-07-13 11:11:47.225490: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-07-13 11:11:47.225609: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [7]:
x = Conv2D(32, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)

In [8]:
avg_p = GlobalAveragePooling2D()(x)
max_p = GlobalMaxPooling2D()(x)
d1 = Dense(4, activation='relu', use_bias=False)
d2 = Dense(32, use_bias=False)

In [9]:
ch_att = Add()([d2(d1(avg_p)), d2(d1(max_p))])
ch_att = Activation('sigmoid')(ch_att)
ch_att = Reshape((1, 1, 32))(ch_att)
x = Multiply()([x, ch_att])

In [10]:
sp_att = Conv2D(1, (7, 7), padding='same', activation='sigmoid', use_bias=False)(x)
x = Multiply()([x, sp_att])
x = MaxPooling2D((2, 2))(x)

In [11]:
sp_att = Conv2D(1, (7, 7), padding='same', activation='sigmoid', use_bias=False)(x)
x = Multiply()([x, sp_att])
x = MaxPooling2D((2, 2))(x)

In [12]:
x = Conv2D(64, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)

In [13]:
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)

In [14]:
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

In [15]:
outputs = Dense(1, activation='sigmoid')(x)
heavy_model = Model(inputs, outputs)

In [16]:
heavy_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc'), 'accuracy']
)

In [17]:
checkpoint = ModelCheckpoint(
    'best_arich_bhabha_model.h5', 
    monitor='val_auc', 
    save_best_only=True, 
    mode='max', verbose=1
)

In [18]:
early_stop = EarlyStopping(
    monitor='val_auc', 
    patience=15, 
    mode='max', 
    restore_best_weights=True, verbose=1
)

In [19]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=5, 
    min_lr=1e-6, verbose=1
)

In [20]:
callbacks_list = [
    checkpoint, 
    early_stop, 
    reduce_lr
]

In [22]:
history = heavy_model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=200, batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks_list, verbose=1
)

Epoch 1/200


2026-07-13 11:16:34.785782: I external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:326] ptxas warning : Registers are spilled to local memory in function 'fusion_137', 32 bytes spill stores, 36 bytes spill loads



2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7841 - auc: 0.8459 - loss: 0.4958

2026-07-13 11:18:02.529855: I external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:326] ptxas warning : Registers are spilled to local memory in function 'fusion_136', 20 bytes spill stores, 20 bytes spill loads



2744/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7841 - auc: 0.8459 - loss: 0.4957
Epoch 1: val_auc improved from None to 0.87112, saving model to best_arich_bhabha_model.h5


...

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.7867 - auc: 0.8683 - loss: 0.4537 - val_accuracy: 0.7971 - val_auc: 0.8748 - val_loss: 0.4370 - learning_rate: 0.0010
Epoch 9/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7918 - auc: 0.8699 - loss: 0.4497
Epoch 9: val_auc improved from 0.87482 to 0.87547, saving model to best_arich_bhabha_model.h5


2744/2744 ━━━━━━━━━━━━━━━━━━━━ 88s 32ms/step - accuracy: 0.7881 - auc: 0.8682 - loss: 0.4535 - val_accuracy: 0.8021 - val_auc: 0.8755 - val_loss: 0.4151 - learning_rate: 0.0010
Epoch 10/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7903 - auc: 0.8698 - loss: 0.4550
Epoch 10: val_auc did not improve from 0.87547

Epoch 10: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.7901 - auc: 0.8703 - loss: 0.4505 - val_accuracy: 0.7989 - val_auc: 0.8745 - val_loss: 0.4335 - learning_rate: 0.0010
Epoch 11/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7937 - auc: 0.8742 - loss: 0.4437
Epoch 11: val_auc improved from 0.87547 to 0.87619, saving model to best_arich_bhabha_model.h5


2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.7921 - auc: 0.8721 - loss: 0.4480 - val_accuracy: 0.7902 - val_auc: 0.8762 - val_loss: 0.4308 - learning_rate: 5.0000e-04
Epoch 12/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7913 - auc: 0.8731 - loss: 0.4469
Epoch 12: val_auc improved from 0.87619 to 0.87722, saving model to best_arich_bhabha_model.h5


2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.7924 - auc: 0.8734 - loss: 0.4462 - val_accuracy: 0.8033 - val_auc: 0.8772 - val_loss: 0.4375 - learning_rate: 5.0000e-04
Epoch 13/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7907 - auc: 0.8758 - loss: 0.4385
Epoch 13: val_auc did not improve from 0.87722
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.7911 - auc: 0.8736 - loss: 0.4452 - val_accuracy: 0.7861 - val_auc: 0.8766 - val_loss: 0.4624 - learning_rate: 5.0000e-04
Epoch 14/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7987 - auc: 0.8798 - loss: 0.4336
Epoch 14: val_auc did not improve from 0.87722
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 86s 31ms/step - accuracy: 0.7927 - auc: 0.8747 - loss: 0.4436 - val_accuracy: 0.7849 - val_auc: 0.8761 - val_loss: 0.4656 - learning_rate: 5.0000e-04
Epoch 15/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.7965 - auc: 0.8779 - loss: 0.4397
Epoch 15: val_auc did not improve from 0.87722



2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.8005 - auc: 0.8787 - loss: 0.4372 - val_accuracy: 0.8156 - val_auc: 0.8778 - val_loss: 0.3970 - learning_rate: 2.5000e-04
Epoch 20/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7993 - auc: 0.8799 - loss: 0.4368
Epoch 20: val_auc did not improve from 0.87782
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.8006 - auc: 0.8801 - loss: 0.4349 - val_accuracy: 0.7978 - val_auc: 0.8772 - val_loss: 0.4432 - learning_rate: 2.5000e-04
Epoch 21/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8002 - auc: 0.8828 - loss: 0.4284
Epoch 21: val_auc did not improve from 0.87782
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 87s 32ms/step - accuracy: 0.7977 - auc: 0.8796 - loss: 0.4361 - val_accuracy: 0.8039 - val_auc: 0.8760 - val_loss: 0.4321 - learning_rate: 2.5000e-04
Epoch 22/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8024 - auc: 0.8811 - loss: 0.4332
Epoch 22: val_auc did not improve from 0.87782
2